# RONA

> Risk of non-adaptedness

The Risk of non-adaptedness (RONA) was first developed by @Rellstab_2016. 

TODO!

In [ ]:
#| default_exp RONA

In [ ]:
#| hide
from fastcore.utils import *
from nbdev.showdoc import *
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
#| export
class RONA:
    "Risk of non-adaptedness genomic offset statistic."
    def __init__(self): 
        self._reg = LinearRegression(copy_X=True, fit_intercept=True)
    def __str__(self):
        return "RONA model."
    __repr__ = __str__

In order to use the model we have first to initialize it.  

In [ ]:
model = RONA()

Then, we have to fit the model (that is, find the the least squares solutions) to

$$
\mathbf y \approx \mathbf x \mathbf b + \mathbf w
$$

where $\mathbf y = [y_1, \dots, y_L], y_l \in \mathbb [0, 1]$ is a vector with the encoded allele frequencies and $\mathbf x = [x_1, \dots, x_P], x_p \in \mathbb R$ is a vector with the environmental covariates. TODO: add details. 

In [ ]:
#|export
@patch
def fit(self:RONA,
        Y: np.ndarray, # Allele frequency matrix (nxL)
        X: np.ndarray): # Environmental matrix (nxP)
    "Fits the RONA model. "
    n1, L = Y.shape
    n2, P = X.shape
    if n1 != n2: 
        raise ValueError("Dimensions of array don't match")
    self._reg.fit(X, Y)

The `fit()` method expects as input an allele matrix $\mathbf Y$ and an environmental matrix $\mathbf X$ with as many rows as individuals. For now, let us simulate them under the ¿generative model?: 

In [ ]:
N, L, P = 100, 10_000, 3_000
rng = np.random.default_rng(1000)
X = rng.normal(loc=0.0, scale=1.0, size=(N, P))
p = rng.uniform(low=0, high=1, size = (1, L))
B = rng.normal(loc=0.0, scale=0.1, size = (P, L))
Y = X@B + np.ones((N, 1))@p
Y = Y.clip(0, 1)
assert X.shape == (N, P)
assert Y.shape == (N, L)

In [ ]:
indices = rng.permutation(N)
training_idx, test_idx = indices[:60], indices[60:]
Xtrain, Xpredict = X[training_idx,:], X[test_idx,:]
Ytrain, Ypredict = Y[training_idx,:], Y[test_idx,:]
model.fit(Ytrain, Xtrain)

Now, we can make predictions: 

In [ ]:
#| export 
@patch
def predict(self:RONA,
        X: np.ndarray # Environmental matrix (nxP)
           )-> np.ndarray: # Predicted allele frequencies
    "Predicts the allele frequencies for a given environmental matrix. "    
    return self._reg.predict(X)


In [ ]:
mse = (np.square(model.predict(Xpredict) - Ypredict)).mean()
mse

np.float64(0.2400697200513574)

Finally, we can predict the genomic offset under two different environments: 

In [ ]:
#| export 
@patch
def genomic_offset(self:RONA,
        X: np.ndarray, # Environmental matrix (nxP)
        Xstar: np.ndarray, # Altered environmental matrix (nxP)
           )-> np.ndarray: # A vector of genomic offsets (n)
    "Predicts the allele frequencies for a given environmental matrix. "
    L = model._reg.coef_.shape[0]
    if X.shape != Xstar.shape:
        raise ValueError("Dimensions of array don't match")
    return np.sum(np.abs(self._reg.predict(X) - self._reg.predict(Xstar)), axis=1) / L

As expected, the genomic offset is zero if both environmental matrixes are identical: 

In [ ]:
model.genomic_offset(Xpredict, Xpredict)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0.])

In [ ]:
offset = model.genomic_offset(Xpredict, Xpredict+rng.normal(size=Xpredict.shape))
offset

array([0.0561077 , 0.04938815, 0.04892588, 0.05394206, 0.04902184,
       0.05374755, 0.05781168, 0.06065314, 0.05513752, 0.04787735,
       0.07143362, 0.05549107, 0.05420392, 0.05473025, 0.06459339,
       0.06245881, 0.05042245, 0.04561041, 0.05867662, 0.05397891,
       0.05450543, 0.05987062, 0.05533189, 0.06458994, 0.05973618,
       0.06524648, 0.05743896, 0.06795648, 0.05891648, 0.05378579,
       0.06035221, 0.05860062, 0.04970914, 0.05667626, 0.0708876 ,
       0.0551229 , 0.0572842 , 0.05130077, 0.06539211, 0.05887068])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()